### Choose where this notebook runs

This is the **IQM edition**: every measurement example is also executed on the IQM backend chosen below — the
local noisy model of **Garnet** (20 qubits) or, via IQM Resonance, **Garnet** / **Emerald** (54 qubits) — after
transpilation to the IQM native gate set **PRX + CZ**; two IQM rules about classical bits show up along the way. Pick the backend
by editing the single `TARGET` line below (see `README.md` and `iqm_backend.py` in this folder):

| `TARGET` | runs on | token? | QPU credits? |
|---|---|---|---|
| `"fake_apollo"` | local 20-qubit noisy simulator | no | no |
| `"garnet:mock"` / `"emerald:mock"` | IQM Resonance cloud mock (pipeline check: returns RANDOM bitstrings, no physics) | yes | no |
| `"garnet"` / `"emerald"` | **real QPU** (20q / 54q) | yes | yes |


# Measurement in Qiskit

**Course**: Quantum Computing Laboratory
**Version**: 2.2 (Qiskit 2.x Compatible)
**Last Updated**: October 2026

---


In [ ]:
from iqm_backend import get_backend, run_counts   # helper module in this folder (see iqm_backend.py)
# run_counts(circuit, TARGET, shots=...) = transpile to the IQM native gates (PRX + CZ) + run + get_counts()
# get_backend(TARGET) returns the raw Qiskit backend object if you want to call backend.run yourself
TARGET = "fake_apollo"   # change this one line to choose the backend (see the table above)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `from iqm_backend import get_backend, run_counts` | the helper module of this folder: `run_counts(circuit, TARGET, shots=...)` = `transpile_to_IQM` (rewrite the circuit with the native `r` = PRX rotations and `cz` gates, routed on the chip's square lattice) → `backend.run` → counts as a plain dict; `run_counts(..., return_transpiled=True)` also returns the transpiled circuit |
| `TARGET = "fake_apollo"` | a local noisy simulator with the 20-qubit Garnet topology: no token, no credits, realistic gate and readout noise; `"garnet:mock"` checks the cloud pipeline but returns RANDOM bitstrings; `"garnet"` / `"emerald"` are the real QPUs (token + credits; IQM Resonance declines circuits above 10 000 native instructions) |
| `get_backend(TARGET)` | the raw Qiskit backend object (name, qubit count, coupling map) if you want to call `backend.run` yourself |

</details>


## 📚 Overview

A quantum circuit gives classical answers only through **measurement**. This notebook collects the four ways the
Qiskit SDK offers to put measurements into a circuit (the examples come from the Qiskit documentation) and, unlike
the documentation, *runs* every circuit so that you see the bitstrings each one produces.

### Topics Covered:
- `QuantumCircuit.measure(qubits, clbits)` — measure chosen qubits into chosen classical bits
- the `Measure` instruction, appended by hand
- `QuantumCircuit.measure_all()` — a barrier plus a new classical register `meas`
- `QuantumCircuit.measure_active()` — only the qubits that are not idle
- reading the result strings: bit ordering and one space-separated group per classical register
- the Born rule: measuring a superposition


## 🎯 Learning Objectives

1. ✅ Map qubits to classical bits with `measure`
2. ✅ Know which classical register each measurement method writes to
3. ✅ Read a counts dictionary: bit 0 on the right, one group per register
4. ✅ Predict measurement probabilities with the Born rule and compare them with sampled counts


## 🔧 Prerequisites

**Required Knowledge:**
- Basic circuit construction (`02_Construct_Circuits.ipynb`)
- State vectors and operators (`04_Operators.ipynb`)

**Required Software:**
- Python 3.9+
- Qiskit 2.x, Qiskit Aer

**Recommended Previous Notebooks:**
- `02_Construct_Circuits.ipynb`, `04_Operators.ipynb`


In [ ]:
from qiskit import QuantumCircuit, ClassicalRegister     # circuits and classical registers
from qiskit.circuit import Measure                       # the measurement instruction as an object (section 2)
from qiskit.quantum_info import Statevector              # the exact state -> exact Born-rule probabilities
from qiskit.visualization import plot_histogram          # bar chart of measurement counts
import numpy as np                                       # pi, sqrt
from IPython.display import display                      # show a figure (e.g. a circuit drawing) from the middle of a cell
def draw(qc, fold=-1, **kw): return qc.draw("mpl", style="clifford", fold=fold, **kw)   # every circuit picture: matplotlib drawer; fold=-1 = one long row, fold=N wraps after N columns

backend = get_backend(TARGET)                            # the IQM backend chosen in the TARGET cell (here only to describe it)
print(f"backend: {backend.name}, {backend.num_qubits} qubits, coupling map with {len(backend.coupling_map.get_edges())} directed edges")
def run_circuit(circuit, shots=1024):
    """Run `circuit` on TARGET and return the counts {bitstring: number of shots}."""
    return run_counts(circuit, TARGET, shots=shots)      # transpile_to_IQM (PRX + CZ) -> backend.run -> get_counts()

print("✓ imports done — circuits run on", TARGET)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `get_backend(TARGET)` | the IQM backend object of the TARGET cell; we only print its name, qubit count and coupling map (pairs of qubits joined by a CZ coupler — each undirected edge appears twice) |
| `def run_circuit(circuit, shots=1024)` | our one-line execution helper: `run_counts` transpiles the circuit to the native `r` (PRX) + `cz` gates on the chip's square lattice, runs it on `TARGET` and returns a dict `{bitstring: count}`. Every later cell calls it |
| `fake_apollo` = noisy | unlike an ideal simulator, a definite state does not give the same bitstring in every shot: gate and readout errors put a few per cent of the shots on wrong outcomes — keep this in mind when reading the counts below |
| `draw(qc)` | our helper around `qc.draw("mpl", style="clifford")`; `display(...)` shows the picture from anywhere in a cell |

</details>


## 1. `QuantumCircuit.measure` method

Use the measure method to measure a QuantumCircuit.

Examples:


In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(5, 5)                      # 5 qubits and 5 classical bits; qubits start in |0>, bits in 0
qc.x(0)                                        # X = quantum NOT: qubit 0 -> |1>
qc.x(1)                                        # qubit 1 -> |1>
qc.x(4)                                        # qubit 4 -> |1>; the state is now |10011> (qubit 4 written first)
qc.measure(range(5), range(5)) #  Measures all qubits into the corresponding clbit.   (pairs element-wise: qubit i -> clbit i)
display(draw(qc)); print("↑ three X gates, then five measurements: each meter sends qubit i down to classical bit i")

counts = run_circuit(qc)                       # 1024 shots
print("counts:", counts)
top = max(counts, key=counts.get)              # the most frequent bitstring (max over the keys, ranked by their counts)
print(f"most frequent result {top!r} = {int(top, 2)} in binary; read right to left: c0 = {top[-1]}, c1 = {top[-2]}, c2 = {top[-3]}, c3 = {top[-4]}, c4 = {top[0]}")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `QuantumCircuit(5, 5)` | 5 qubits `q_0…q_4` and one classical register `c` of 5 bits: measurements need classical bits to write into |
| `qc.measure(range(5), range(5))` | two sequences paired element by element: qubit 0 → clbit 0, …, qubit 4 → clbit 4. A single pair `measure(1, 0)` is the next cell |
| `counts` → `{'10011': …}` | the keys are the classical bits written **c4 c3 c2 c1 c0**: bit 0 is the *rightmost* character (little-endian, like a binary number). So qubits 0, 1 and 4 — the flipped ones — appear as the 1s at positions 0, 1, 4 from the right |
| `max(counts, key=counts.get)`, `int(top, 2)` | the most frequent key; `int(s, 2)` reads a bitstring as a base-2 integer: 0b10011 = 16 + 2 + 1 = 19 |

</details>


<details open><summary><b>📐 Working it out</b></summary>

**Born rule for a basis state.** After the three X gates the register is in the computational-basis state
$|\psi\rangle = |q_4q_3q_2q_1q_0\rangle = |1\,0\,0\,1\,1\rangle$. The probability of reading the bitstring $b$ is
$P(b)=|\langle b|\psi\rangle|^2$, which is $1$ for $b = 10011$ and $0$ for the other 31 strings: a basis state gives a
**certain** outcome, and every one of the 1024 shots of an ideal simulator returns `'10011'`.

**Bit ordering.** Qiskit prints classical bit $c_0$ on the right, as in a binary number:
$10011_2 = 1\cdot 2^4 + 0\cdot 2^3 + 0\cdot 2^2 + 1\cdot 2^1 + 1\cdot 2^0 = 16+2+1 = 19$, the integer printed by the cell.
Reading the string left to right as "qubit 0, qubit 1, …" would wrongly suggest that qubits 0, 3 and 4 were flipped.

</details>


In [ ]:
# What the IQM chip actually executes: the transpiled version of the 5-qubit circuit
counts_iqm, qc_iqm = run_counts(qc, TARGET, shots=1024, return_transpiled=True)   # also return the circuit after transpile_to_IQM
print(f"native instructions: {dict(qc_iqm.count_ops())}, depth {qc_iqm.depth()}")    # 'r' = PRX rotation: each X became r(π, 0)
print(f"P('10011') = {counts_iqm.get('10011', 0) / 1024:.3f} on {TARGET} (exactly 1 on an ideal machine); "
      f"{len(counts_iqm)} different bitstrings were observed")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `run_counts(qc, TARGET, ..., return_transpiled=True)` | the same pipeline as `run_circuit`, but it also hands back the circuit after `transpile_to_IQM`: what the chip really executes |
| `count_ops()` → `r`, `measure` | the three X gates became three `r` = PRX(θ = π, φ = 0) rotations — PRX is the only single-qubit gate of IQM chips — and the five measurements stay as they are; no `cz` is needed because the circuit has no two-qubit gate |
| `P('10011')` < 1 | on `fake_apollo` some shots end on other strings: each of the five readouts has an error of a few per cent, and a wrong bit anywhere changes the string. On `garnet` / `emerald` this is the readout fidelity of the real chip |

</details>


In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)                      # 3 qubits but only ONE classical bit
qc.x([0, 2])                                   # a list applies X to each qubit in it: the state is |101>
qc.measure(1, 0)    # Measure qubit 1 into the classical bit 0.   measure(qubit, clbit)
display(draw(qc)); print("↑ only qubit 1 is measured, into the single classical bit c0; qubits 0 and 2 are flipped but never read")

counts = run_circuit(qc)
print("counts:", counts)                       # one-character keys: there is one classical bit, and qubit 1 is in |0>


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `QuantumCircuit(3, 1)` | the numbers of qubits and of classical bits are independent: you only need as many bits as you want to read |
| `qc.x([0, 2])` | most gate methods accept a list of qubits and apply the gate to each one |
| `qc.measure(1, 0)` | measure(qubit, clbit): qubit 1, which was never flipped, is read into bit 0 — so the key is `'0'`. The X gates on qubits 0 and 2 have no effect on the result: an unmeasured qubit leaves no trace in the counts |

</details>


## 2. `Measure` class

The Qiskit Measure class measures the specified qubits.


In [ ]:
from qiskit.circuit import Measure

#.... (we continue the circuit qc of the previous cell: 3 qubits, 1 clbit, qubit 1 already measured into c0)
qc.append(Measure(), [0], [0])  # measure qubit 0 into clbit 0     append(instruction, [qubits], [clbits])
display(draw(qc)); print("↑ a second meter writes qubit 0 into the SAME classical bit c0, after qubit 1 was written there")
print("instructions in order:", [inst.operation.name for inst in qc.data])   # qc.data = the list of instructions

try:                                           # IQM backends require every measurement to have its own classical bit
    counts = run_circuit(qc)
    print("counts:", counts)
except Exception as err:                       # CircuitValidationError: non-unique measurement key
    print(f"{TARGET} refuses this circuit -> {type(err).__name__}: {err}")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qc.append(Measure(), [0], [0])` | the long form of `qc.measure(0, 0)`: create the instruction object, then attach it to a list of qubits and a list of classical bits. `append` works the same way for any gate or sub-circuit |
| `qc.data` | the circuit as a list of instructions in time order: `['x', 'x', 'measure', 'measure']` here (`x([0, 2])` appended one X per qubit) |
| `try: … except Exception as err:` | the IQM backends turn each measurement into a *measurement key* named after its classical bit and require the keys to be unique: two measurements into c0 raise `CircuitValidationError: … non-unique measurement key`. On an ideal Qiskit simulator the second measurement simply overwrites c0 (result `'1'`). On IQM, give every measurement its own classical bit |

</details>


## 3. `QuantumCircuit.measure_all` method

To measure all qubits into the corresponding classical bits, use the measure_all method. By default, this method adds new classical bits in a ClassicalRegister to store these measurements.


In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)                      # the 1-bit register 'c' declared here will stay unused
qc.x([0, 2])                                   # |101>
qc.measure_all() # Measure all qubits.          adds a barrier + a NEW 3-bit register 'meas', then qubit i -> meas[i]
display(draw(qc)); print("↑ measure_all(): a barrier (dashed line), then three meters writing into the new register 'meas'; register c is never written")
print("classical registers:", [(creg.name, creg.size) for creg in qc.cregs])   # qc.cregs = the list of ClassicalRegister objects

counts = run_circuit(qc)
print("counts:", counts)                       # keys 'meas c': one group per register, separated by a space
for key, n in counts.items():
    groups = key.split(" ")                    # e.g. '101 0' -> ['101', '0'] = [register meas, register c]
    print(f"  key {key!r}: meas = {groups[0]!r}, c = {groups[1]!r} -> {n} shots")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qc.measure_all()` | three things at once: a `barrier` over all qubits, a new `ClassicalRegister(3, 'meas')`, and one measurement per qubit into it. The register `c` from the constructor is *not* used — pass `add_bits=False` to reuse existing bits instead (there must be at least as many as qubits) |
| `qc.cregs` | the classical registers of the circuit, in creation order: `[('c', 1), ('meas', 3)]` |
| key `'101 '` | with several registers the key has one group per register, separated by a space, the **last-created register first**: `meas` = `101` (qubits 2, 0 flipped), then `c`. An ideal Qiskit simulator prints the never-written `c` as `0` (`'101 0'`); the IQM backends return only the bits that were measured, so the group of `c` is **empty** and the key ends with a space. Other keys (`'001 '`, `'111 '`, …) are noise |
| `key.split(" ")` | the safe way to pull the registers apart (it works for both forms of the key); inside each group bit 0 is still the rightmost character |

</details>


<details open><summary><b>📐 Working it out</b></summary>

**What `measure_all()` adds.** For a circuit with $n$ qubits it appends (i) a `barrier` on all $n$ qubits — a fence
that the transpiler may not move gates across, drawn as a dashed line — (ii) a new register `meas` of $n$ bits, and
(iii) the $n$ measurements $q_i \to \texttt{meas}[i]$. Here $n=3$ and the circuit gets 1 barrier + 3 measurements.

**Reading the key.** The state is $|q_2q_1q_0\rangle=|101\rangle$, so with probability $1$ the register `meas` holds
$\texttt{meas}[2]\,\texttt{meas}[1]\,\texttt{meas}[0] = 1\,0\,1$. Registers are printed last-created first, so the key is
`meas` followed by `c`: `'101 0'`, where the trailing `0` is the untouched bit of `c`. The same rule explains why
`counts.get('101')` would return nothing: the key contains the space and the second group. (The IQM backends return the never-written register `c` empty, so there the key is `'101 '`.)

</details>


## 4. `QuantumCircuit.measure_active` method

To measure all qubits that are not idle, use the measure_active method. This method creates a new ClassicalRegister with a size equal to the number of non-idle qubits being measured.


In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure_active() # Measure qubits that are not idle, i.e., qubits 0 and 2.   (new register sized to the active qubits)
display(draw(qc)); print("↑ measure_active(): qubit 1 has no gate, so only qubits 0 and 2 get a meter, into a new 2-bit register 'meas'")
print("classical registers:", [(creg.name, creg.size) for creg in qc.cregs])

counts = run_circuit(qc)
print("counts:", counts)                       # meas[0] = qubit 0, meas[1] = qubit 2: the idle qubit 1 is skipped


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qc.measure_active()` | like `measure_all()` (barrier + new register named `meas`), but only for the qubits that carry at least one instruction; the new register has 2 bits, one per active qubit |
| `meas[0]` ← qubit 0, `meas[1]` ← qubit 2 | the bits are assigned to the active qubits in order, so the bit *index* no longer equals the qubit index: the key `'11 '` means qubits 0 and 2 both read 1 (the empty group after the space is the unused register `c`, as in the previous cell) |
| when to use it | a quick way to skip ancillas or unused wires — but check `qc.cregs` (or the drawing) to know which bit is which qubit |

</details>


## 5. Measuring a superposition: the Born rule

Every circuit so far prepared a computational-basis state, so each measurement had one certain outcome. Measurement
becomes probabilistic as soon as the state is a superposition: for $|\psi\rangle=\sum_b \alpha_b\,|b\rangle$ the
bitstring $b$ is observed with probability $P(b)=|\alpha_b|^2$ (**Born rule**), and the counts only *estimate* these
probabilities. Below we compute them exactly with `Statevector` and compare with 4096 shots.


In [ ]:
qc = QuantumCircuit(2)                         # no classical bits: measure_all() will create them
qc.h(0)                                        # qubit 0 -> (|0> + |1>)/sqrt2
qc.ry(2 * np.pi / 3, 1)                        # qubit 1 -> cos(pi/3)|0> + sin(pi/3)|1>   (Ry(θ) rotates by θ about the y axis)
exact = Statevector(qc).probabilities_dict()   # Born rule on the exact state; must be computed BEFORE adding measurements
qc.measure_all()
display(draw(qc)); print("↑ H on qubit 0, Ry(2π/3) on qubit 1, then measure_all(): the only register is 'meas', so the keys have no space")

shots = 4096
counts = run_circuit(qc, shots=shots)
print(f"{'outcome':>8} {'Born P':>8} {'measured':>9}")
for b in sorted(exact):                        # '00', '01', '10', '11'  = q1 q0
    print(f"{b:>8} {exact[b]:8.4f} {counts.get(b, 0) / shots:9.4f}")   # counts.get(b, 0): 0 if b never occurred
display(plot_histogram(counts)); print("↑ the four outcomes of 4096 shots: 10 and 11 about three times as frequent as 00 and 01")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qc.ry(2 * np.pi / 3, 1)` | a rotation by θ = 2π/3 about y: \|0⟩ → cos(θ/2)\|0⟩ + sin(θ/2)\|1⟩ = ½\|0⟩ + (√3/2)\|1⟩, so P(qubit 1 = 1) = 3/4 |
| `Statevector(qc).probabilities_dict()` | the exact amplitudes of the circuit's final state, squared: the Born-rule probabilities keyed like the counts. It must be taken before `measure_all()` — a `Statevector` cannot be built from a circuit with measurements |
| `counts.get(b, 0) / shots` | the relative frequency of each outcome: close to the Born probability, but not equal — the difference is *shot noise*, of order √(P(1−P)/N) ≈ 0.007 here |
| `plot_histogram(counts)` | the bar chart of the sampled counts; with `display` it can be shown before the end of the cell |

</details>


<details open><summary><b>📐 Working it out</b></summary>

The two qubits are prepared independently, so the state is a product and the probabilities multiply:
$$|\psi\rangle = \Big(\tfrac12|0\rangle+\tfrac{\sqrt3}{2}|1\rangle\Big)_{q_1}\otimes\tfrac{1}{\sqrt2}\big(|0\rangle+|1\rangle\big)_{q_0},
\qquad P(q_1q_0) = |\alpha_{q_1}|^2\,|\alpha_{q_0}|^2 .$$
With $|\alpha_{q_0}|^2=\tfrac12$ for both values and $|\alpha_{q_1=0}|^2=\tfrac14$, $|\alpha_{q_1=1}|^2=\tfrac34$:
$$P(00)=P(01)=\tfrac14\cdot\tfrac12=0.125,\qquad P(10)=P(11)=\tfrac34\cdot\tfrac12=0.375 .$$
(The keys are $q_1q_0$: the left character is qubit 1.) A sampled frequency $\hat p$ from $N$ shots has standard
deviation $\sqrt{p(1-p)/N}$; for $p=0.375$, $N=4096$ this is $\sqrt{0.375\cdot0.625/4096}\approx 0.0076$. On
`fake_apollo` the deviations are a little larger: readout errors move a few per cent of the shots from the
frequent outcomes `10`, `11` to the rare ones `00`, `01` (readout of $|1\rangle$ is the less reliable one).

</details>


## 🔬 In-depth analysis

1. **Measuring in the X basis.** Qiskit's `measure` always reads the Z basis ($|0\rangle$, $|1\rangle$). To measure in the X basis ($|+\rangle$, $|-\rangle$), apply H just before the measurement. Prepare $|+\rangle$ (`h`) and $|-\rangle$ (`x`, `h`), and estimate P(0) for each when measured (a) directly and (b) after an extra H.
2. **Correlations of a Bell pair.** Prepare $(|00\rangle+|11\rangle)/\sqrt2$ (`h(0)`, `cx(0, 1)`) and measure qubit 0 into clbit 0 and qubit 1 into clbit 1 with explicit `measure` calls. Show that each bit alone is random (P(1) ≈ 1/2) while the two bits agree in (almost) every shot.
3. **Shot noise.** For the qubit $R_y(2\pi/3)|0\rangle$ of section 5 (P(1) = 3/4), estimate P(1) with N = 100, 1000 and 10 000 shots and compare the error $|\hat p - 0.75|$ with the expected standard deviation $\sqrt{p(1-p)/N}$.

Write your code in the empty cell below; a reference solution follows it. Try first, then compare.


In [ ]:
# 🔬 In-depth analysis — your code here



In [ ]:
# Reference solution

# (1) X-basis measurement: H before measure maps |+> -> |0> and |-> -> |1>
def prepared(minus, extra_h):
    qc = QuantumCircuit(1, 1)
    if minus:
        qc.x(0)                                # |1> ...
    qc.h(0)                                    # ... -> |->;  |0> -> |+>
    if extra_h:
        qc.h(0)                                # rotate the X basis onto the Z basis
    qc.measure(0, 0)
    return qc

for minus in (False, True):
    for extra_h in (False, True):
        c = run_circuit(prepared(minus, extra_h), shots=2000)
        print(f"(1) state {'|->' if minus else '|+>'}, {'H + measure' if extra_h else 'measure    '}: P(0) = {c.get('0', 0) / 2000:.3f}")

# (2) Bell pair: random bits that always agree
bell = QuantumCircuit(2, 2)
bell.h(0); bell.cx(0, 1)                       # (|00> + |11>)/sqrt2
bell.measure(0, 0); bell.measure(1, 1)         # explicit qubit -> clbit pairs
c = run_circuit(bell, shots=2000)
p_q0 = sum(n for k, n in c.items() if k[-1] == '1') / 2000    # k[-1] = clbit 0 (rightmost)
p_q1 = sum(n for k, n in c.items() if k[-2] == '1') / 2000    # k[-2] = clbit 1
agree = (c.get('00', 0) + c.get('11', 0)) / 2000
print(f"(2) P(c0 = 1) = {p_q0:.3f}, P(c1 = 1) = {p_q1:.3f}, P(c0 = c1) = {agree:.3f}   (ideal: 0.5, 0.5, 1)")

# (3) shot noise of the estimate of P(1) = 3/4
ry_qc = QuantumCircuit(1, 1)
ry_qc.ry(2 * np.pi / 3, 0); ry_qc.measure(0, 0)
for N in (100, 1000, 10000):
    p_hat = run_circuit(ry_qc, shots=N).get('1', 0) / N
    sigma = np.sqrt(0.75 * 0.25 / N)           # binomial standard deviation of a frequency
    print(f"(3) N = {N:>5}: p̂ = {p_hat:.4f}, |p̂ - 0.75| = {abs(p_hat - 0.75):.4f}, σ = {sigma:.4f}")
display(draw(bell)); print("↑ task 2: the Bell pair with two explicit measurements, qubit i -> clbit i")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qc.h(0)` before `measure` | a change of basis: H maps \|+⟩ → \|0⟩ and \|−⟩ → \|1⟩, so (b) gives P(0) ≈ 1 for \|+⟩ and ≈ 0 for \|−⟩, while a direct Z measurement (a) gives 0.5 for both — it cannot tell them apart |
| `k[-1]`, `k[-2]` | indexing from the right of the key picks clbit 0 and clbit 1 (little-endian strings) |
| `P(c0 = c1)` ≈ 1 | each half of a Bell pair alone is a fair coin, yet the two coins always agree: the correlation is in the joint distribution, which only the full key shows |
| `np.sqrt(0.75 * 0.25 / N)` | the binomial standard deviation √(p(1−p)/N): 0.043, 0.014, 0.0043 for N = 100, 1000, 10 000. The error shrinks like 1/√N — 100× more shots for one more decimal digit. On a noisy backend it stops shrinking at the level of the systematic (readout) bias, a few 0.01 on `fake_apollo`: more shots do not remove systematic errors |

</details>


## 📖 References

### Documentation

1. **Measure qubits** (Qiskit guide):
   https://quantum.cloud.ibm.com/docs/guides/measure-qubits

2. **`QuantumCircuit.measure`, `measure_all`, `measure_active`** (API reference):
   https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.QuantumCircuit

3. **Bit ordering in the Qiskit SDK**:
   https://quantum.cloud.ibm.com/docs/guides/bit-ordering

---

**Course**: Quantum Computing Laboratory
**License**: MIT
**Version**: 2.2.0
**Last Updated**: October 2026
